<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/03_construction_eda_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การสำรวจโครงการจ้างก่อสร้างและการกำหนดกลุ่มศึกษา ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

Notebook นี้ตอบคำถามตามลำดับดังนี้:

1. โครงการก่อสร้างส่วนใหญ่อยู่ในช่วงวงเงินใด
2. การกระจุกของวงเงินสัมพันธ์กับวิธีจัดซื้อใด
3. กฎหมายอธิบายการกระจุกดังกล่าวได้อย่างไร
4. ควรกำหนดกลุ่มศึกษาสำหรับค้นหา Pattern อย่างไร


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from matplotlib.patches import FancyBboxPatch

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [ ]:
base_dir = Path('/content/drive/MyDrive/learning/dads/dads5001/project_1_dads5001/dataset/procurement/egp-contract')

data_path = base_dir / 'processed' / 'construction_contracts_2569.csv'
project_output_path = base_dir / 'processed' / 'construction_projects_2569.csv'

project_dir = base_dir.parents[2]
figure_dir = project_dir / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'Input file: {data_path}')
print(f'Project output: {project_output_path}')
print(f'Figure directory: {figure_dir}')

In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
MUTED = '#667085'
GRID = '#E4E7EC'

## 1. สร้างข้อมูลระดับโครงการ

ไฟล์จาก Notebook 02 อยู่ในระดับ `โครงการ–สัญญา–บริษัท` การสำรวจวงเงินและวิธีจัดซื้อใน Notebook นี้ต้องใช้หนึ่งแถวต่อ `รหัสโครงการ`


In [ ]:
construction_data = pd.read_csv(data_path, low_memory=False)

print(f'รายการจาก Notebook 02: {len(construction_data):,}')
print(f'จำนวนคอลัมน์: {construction_data.shape[1]:,}')

display(construction_data.head())

In [ ]:
project_id_column = 'รหัสโครงการ'
budget_column = 'วงเงินงบประมาณ (บาท)'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'

project_columns = [
    project_id_column,
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อประเภทโครงการ',
    'ชื่อหน่วยงาน',
    'ชื่อหน่วยงานย่อย',
    method_column,
    'ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    budget_column,
    'ราคากลาง (บาท)',
    'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)',
    'ปีงบประมาณ',
    'วันที่เกิดรายการ',
    'จังหวัด',
    'เขต/อำเภอ',
    'แขวง/ตำบล',
    'สถานะโครงการ',
    'พิกัดของโครงการ',
    'ละติจูดของโครงการ',
    'ลองจิจูดของโครงการ'
]

In [ ]:
project_data = construction_data[project_columns].drop_duplicates(subset=project_id_column,keep='first').copy()

project_data[budget_column] = pd.to_numeric(project_data[budget_column], errors='coerce')

print(f'รายการระดับสัญญา–บริษัท: {len(construction_data):,}')
print(f'โครงการไม่ซ้ำ: {len(project_data):,}')
print(f'แถวที่รวมเป็นระดับโครงการ: {len(construction_data) - len(project_data):,}')

ข้อมูลสำหรับ EDA เหลือหนึ่งแถวต่อโครงการ ส่วนข้อมูลผู้รับจ้างยังอยู่ในไฟล์จาก Notebook 02 และจะใช้เลขประจำตัวนิติบุคคลเป็นคีย์ใน Notebook 04


## 2. โครงการก่อสร้างส่วนใหญ่อยู่ช่วงใด

เริ่มจากดูค่ากลาง ช่วงวงเงิน และวงเงินที่พบซ้ำบ่อย โดยยังไม่ใช้กฎหมายเป็นจุดตั้งต้น


In [ ]:
budget_summary = project_data[budget_column].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])

display(budget_summary.to_frame(name='วงเงินงบประมาณ'))

print(f'Mean: {project_data[budget_column].mean():,.0f} บาท')
print(f'Median: {project_data[budget_column].median():,.0f} บาท')
print(f'99th percentile: {project_data[budget_column].quantile(0.99):,.0f} บาท')

In [ ]:
common_budget_values = (
    project_data[budget_column]
    .value_counts()
    .head(15)
    .rename_axis('วงเงินงบประมาณ')
    .reset_index(name='จำนวนโครงการ')
)

common_budget_values['สัดส่วนโครงการ (%)'] = (common_budget_values['จำนวนโครงการ'] / len(project_data) * 100)

display(common_budget_values)

In [ ]:
budget_bins = [
    0,
    100_000,
    200_000,
    300_000,
    400_000,
    500_000,
    1_000_000,
    5_000_000,
    10_000_000,
    50_000_000,
    np.inf
]

budget_labels = [
    'ไม่เกิน 100,000',
    '100,001–200,000',
    '200,001–300,000',
    '300,001–400,000',
    '400,001–500,000',
    '500,001–1 ล้าน',
    'มากกว่า 1–5 ล้าน',
    'มากกว่า 5–10 ล้าน',
    'มากกว่า 10–50 ล้าน',
    'มากกว่า 50 ล้าน'
]

project_data['budget_band'] = pd.cut(
    project_data[budget_column],
    bins=budget_bins,
    labels=budget_labels,
    include_lowest=True
)

In [ ]:
budget_band_summary = (
    project_data
    .groupby('budget_band', observed=False)
    .agg(
        project_count=(project_id_column, 'size'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

budget_band_summary['project_pct'] = (
    budget_band_summary['project_count']
    / len(project_data)
    * 100
)

budget_band_summary['budget_pct'] = (
    budget_band_summary['total_budget']
    / budget_band_summary['total_budget'].sum()
    * 100
)

display(budget_band_summary)


In [ ]:
plot_data = budget_band_summary.copy()
colors = [
    ORANGE if str(band) == '400,001–500,000' else BLUE
    for band in plot_data['budget_band']
]

fig, ax = plt.subplots(figsize=(10, 6.5))

bars = ax.barh(
    plot_data['budget_band'],
    plot_data['project_pct'],
    color=colors,
    height=0.62
)

ax.bar_label(
    bars,
    labels=[f'{value:.1f}%' for value in plot_data['project_pct']],
    padding=4,
    fontsize=11,
    color=TEXT
)

ax.invert_yaxis()
ax.set_xlim(0, 30)
ax.set_title(
    'สัดส่วนโครงการตามช่วงงบประมาณ',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('สัดส่วนโครงการ (%)')
ax.set_ylabel('ช่วงวงเงินงบประมาณ (บาท)')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.text(
    0.01,
    0.01,
    f'ฐาน: {len(project_data):,} โครงการ | ข้อมูลสะสมถึง 30 กรกฎาคม 2569',
    fontsize=10,
    color=MUTED
)

fig.tight_layout(rect=[0, 0.04, 1, 1])

In [ ]:
png_path = figure_dir / 'fig03_01_project_and_budget_share_by_budget_band.png'
svg_path = figure_dir / 'fig03_01_project_and_budget_share_by_budget_band.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

### สิ่งที่พบ

วงเงินมัธยฐานอยู่ที่ 395,000 บาท ขณะที่ค่าเฉลี่ยประมาณ 2.66 ล้านบาท แสดงว่าการกระจายเบ้ขวาจากโครงการขนาดใหญ่จำนวนน้อย

โครงการไม่เกิน 500,000 บาทรวมกันคิดเป็น 76.34% ของโครงการทั้งหมด และช่วง 400,001–500,000 บาทมีสัดส่วนสูงที่สุดที่ 25.1%

จากภาพรวมนี้ ขั้นต่อไปคือขยายดูบริเวณใกล้ 500,000 บาทว่าการกระจุกเกิดขึ้นในช่วงใด

## 3. โครงการกระจุกบริเวณใดรอบ 500,000 บาท

ช่วง 400,000–550,000 บาทช่วยขยายภาพจากช่วงงบประมาณกว้างให้เห็นตำแหน่งการกระจุกก่อนนำไปเชื่อมกับวิธีจัดซื้อ

In [ ]:
threshold_bands = [
    (400_000, 409_999, '400–409k'),
    (410_000, 419_999, '410–419k'),
    (420_000, 429_999, '420–429k'),
    (430_000, 439_999, '430–439k'),
    (440_000, 449_999, '440–449k'),
    (450_000, 459_999, '450–459k'),
    (460_000, 469_999, '460–469k'),
    (470_000, 479_999, '470–479k'),
    (480_000, 489_999, '480–489k'),
    (490_000, 500_000, '490–500k'),
    (500_001, 510_000, '500–510k'),
    (510_001, 520_000, '510–520k'),
    (520_001, 530_000, '520–530k'),
    (530_001, 540_000, '530–540k'),
    (540_001, 550_000, '540–550k')
]

threshold_results = []

for lower, upper, label in threshold_bands:
    project_count = project_data[budget_column].between(
        lower,
        upper,
        inclusive='both'
    ).sum()

    threshold_results.append({
        'ช่วงวงเงิน': label,
        'จำนวนโครงการ': project_count,
        'ต่ำกว่า/เท่ากับเพดาน': upper <= 500_000
    })

threshold_summary = pd.DataFrame(threshold_results)

display(threshold_summary)


In [ ]:
colors = [
    ORANGE if index == 9 else BLUE if index < 9 else GRAY
    for index in threshold_summary.index
]

fig, ax = plt.subplots(figsize=(10.5, 5.5))

bars = ax.bar(
    threshold_summary['ช่วงวงเงิน'],
    threshold_summary['จำนวนโครงการ'],
    color=colors,
    width=0.72
)

peak_index = threshold_summary['จำนวนโครงการ'].idxmax()
peak_value = threshold_summary.loc[peak_index, 'จำนวนโครงการ']

ax.text(
    peak_index,
    peak_value + 700,
    f'{peak_value:,.0f}',
    ha='center',
    fontsize=11,
    fontweight='semibold',
    color=ORANGE
)

ax.axvline(
    9.5,
    color='#C2413B',
    linestyle='--',
    linewidth=1.4
)

ax.set_title(
    'จำนวนโครงการรอบวงเงิน 500,000 บาท',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('ช่วงวงเงินงบประมาณ (บาท)')
ax.set_ylabel('จำนวนโครงการ')
ax.tick_params(axis='x', rotation=45)

ax.grid(axis='y', color=GRID, linewidth=0.8)
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.text(
    0.01,
    0.01,
    'สีส้ม: 490,000–500,000 บาท | เส้นประ: 500,000 บาท',
    fontsize=10,
    color=MUTED
)

fig.tight_layout(rect=[0, 0.05, 1, 1])

In [ ]:
png_path = figure_dir / 'fig03_02_budget_distribution_around_500k.png'
svg_path = figure_dir / 'fig03_02_budget_distribution_around_500k.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

In [ ]:
near_threshold_windows = [
    (450_000, 500_000, '450,000–500,000'),
    (480_000, 500_000, '480,000–500,000'),
    (490_000, 500_000, '490,000–500,000'),
    (495_000, 500_000, '495,000–500,000')
]

window_results = []

for lower, upper, label in near_threshold_windows:
    project_count = study_project_data[budget_column].between(
        lower,
        upper,
        inclusive='both'
    ).sum()

    window_results.append({
        'ช่วงวงเงิน': label,
        'จำนวนโครงการ': project_count,
        'สัดส่วนกลุ่มศึกษา (%)': (
            project_count / len(study_project_data) * 100
        )
    })

near_threshold_summary = pd.DataFrame(window_results)

display(near_threshold_summary)


### สิ่งที่พบ

ช่วง 490,000–500,000 บาทมี 26,240 โครงการ สูงกว่าช่วงใกล้เคียงอย่างชัดเจน จึงเป็นจุดกระจุกที่ควรตรวจบริบทต่อ

อย่างไรก็ตาม การกระจุกใกล้เส้น 500,000 บาทยังไม่ใช่หลักฐานของความผิดปกติ ขั้นต่อไปจึงต้องตรวจว่าสัมพันธ์กับวิธีจัดซื้อใด และมีกฎหมายอธิบายหรือไม่

## 4. การกระจุกสัมพันธ์กับวิธีจัดซื้อใด

เมื่อพบการกระจุกบริเวณก่อนถึง 500,000 บาท ขั้นต่อไปคือเปรียบเทียบว่าแต่ละวิธีจัดซื้อมีบทบาทต่างกันอย่างไร โดยแยก “จำนวนโครงการ” ออกจาก “วงเงินรวม”

In [ ]:
method_summary = (
    project_data
    .groupby(method_column)
    .agg(
        project_count=(project_id_column, 'size'),
        total_budget=(budget_column, 'sum'),
        median_budget=(budget_column, 'median')
    )
    .reset_index()
)

method_summary['project_pct'] = (method_summary['project_count'] / len(project_data) * 100)
method_summary['budget_pct'] = (method_summary['total_budget'] / method_summary['total_budget'].sum() * 100)
method_summary = method_summary.sort_values('project_count', ascending=False)

display(method_summary)

In [ ]:
method_names = {
    'เฉพาะเจาะจง': 'เฉพาะเจาะจง',
    'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-bidding',
    'คัดเลือก': 'คัดเลือก'
}

method_plot = method_summary[
    method_summary[method_column].isin(method_names)
].copy()

method_plot['method_label'] = (
    method_plot[method_column]
    .map(method_names)
)


In [ ]:
method_plot = method_plot.sort_values(
    'project_count',
    ascending=True
).reset_index(drop=True)

y = np.arange(len(method_plot))
bar_height = 0.30

fig, ax = plt.subplots(figsize=(9.5, 4.8))

project_bars = ax.barh(
    y + bar_height / 2,
    method_plot['project_pct'],
    height=bar_height,
    color=BLUE,
    label='จำนวนโครงการ'
)

budget_bars = ax.barh(
    y - bar_height / 2,
    method_plot['budget_pct'],
    height=bar_height,
    color=ORANGE,
    label='วงเงินรวม'
)

ax.bar_label(
    project_bars,
    labels=[f'{value:.1f}%' for value in method_plot['project_pct']],
    padding=4,
    fontsize=11,
    color=TEXT
)

ax.bar_label(
    budget_bars,
    labels=[f'{value:.1f}%' for value in method_plot['budget_pct']],
    padding=4,
    fontsize=11,
    color=TEXT
)

ax.set_yticks(y)
ax.set_yticklabels(method_plot['method_label'])
ax.set_xlim(0, 90)

ax.set_title(
    'สัดส่วนโครงการและวงเงินรวมตามวิธีจัดซื้อ',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('สัดส่วน (%)')
ax.set_ylabel('วิธีจัดซื้อจัดจ้าง')
ax.legend(frameon=False, loc='lower right')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.tight_layout()

In [ ]:
png_path = figure_dir / 'fig03_03_project_and_budget_share_by_method.png'
svg_path = figure_dir / 'fig03_03_project_and_budget_share_by_method.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

### สิ่งที่พบ

วิธีเฉพาะเจาะจงคิดเป็น 76.4% ของจำนวนโครงการ แต่คิดเป็น 9.3% ของวงเงินรวม ขณะที่ e-bidding มี 21.2% ของจำนวนโครงการ แต่ครอง 83.3% ของวงเงินรวม

ผลนี้แยกให้เห็นชัดว่า “วิธีที่ใช้บ่อยที่สุด” ไม่ใช่ “วิธีที่มีมูลค่ารวมสูงที่สุด” และนำไปสู่คำถามว่าเหตุใดโครงการขนาดเล็กจึงกระจุกอยู่ในวิธีเฉพาะเจาะจง

## 5. กฎหมายอธิบายการกระจุกอย่างไร

เมื่อย้อนผลจากข้อมูลไปเทียบกับกฎหมาย พบว่า:

- พระราชบัญญัติการจัดซื้อจัดจ้างฯ พ.ศ. 2560 มาตรา 55 กำหนดวิธีจัดซื้อจัดจ้างพัสดุ 3 วิธี ได้แก่ วิธีประกาศเชิญชวนทั่วไป วิธีคัดเลือก และวิธีเฉพาะเจาะจง
- มาตรา 56 วรรคหนึ่ง (2)(ข) เปิดให้ใช้วิธีเฉพาะเจาะจงกับพัสดุทั่วไป เมื่อวงเงินต่อครั้งไม่เกินวงเงินที่กำหนดในกฎกระทรวง
- กฎกระทรวงกำหนดวงเงินดังกล่าวไว้ไม่เกิน 500,000 บาท

แหล่งอ้างอิง: [พระราชบัญญัติการจัดซื้อจัดจ้างฯ ในราชกิจจานุเบกษา](https://www.ratchakitcha.soc.go.th/) และ [กฎหมาย/ระเบียบด้านการจัดซื้อจัดจ้างของกรมบัญชีกลาง](https://www.cgd.go.th/)

ดังนั้น การกระจุกของวิธีเฉพาะเจาะจงใต้ 500,000 บาทเป็นรูปแบบเชิงโครงสร้างที่กฎหมายอธิบายได้ ไม่ใช่ anomaly ด้วยตัวเอง

## 6. กำหนดกลุ่มศึกษาหลัก

เพื่อค้นหา Pattern ที่กฎหมายยังอธิบายไม่ได้ จะเปรียบเทียบโครงการภายใต้บริบทเดียวกัน คือวิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาท

In [ ]:
under_500k_mask = project_data[budget_column] <= 500_000

study_population_mask = (under_500k_mask & project_data[method_column].eq('เฉพาะเจาะจง'))

project_data['is_study_population'] = study_population_mask
study_project_data = project_data[study_population_mask].copy()

study_scope = pd.DataFrame({
    'ขั้นการเลือกข้อมูล': [
        'โครงการก่อสร้างทั้งหมด',
        'วงเงินไม่เกิน 500,000 บาท',
        'เฉพาะเจาะจงและไม่เกิน 500,000 บาท'
    ],
    'project_count': [
        len(project_data),
        under_500k_mask.sum(),
        len(study_project_data)
    ]
})

study_scope['project_pct'] = (study_scope['project_count'] / len(project_data) * 100)

display(study_scope)

In [ ]:
stages = study_scope['ขั้นการเลือกข้อมูล']
counts = study_scope['project_count']
percentages = study_scope['project_pct']
colors = [GRAY, GRAY, ORANGE]

fig, ax = plt.subplots(figsize=(10, 4.6))

bars = ax.barh(
    stages,
    counts,
    color=colors,
    height=0.58
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} ({pct:.1f}%)'
        for count, pct in zip(counts, percentages)
    ],
    padding=5,
    fontsize=11,
    color=TEXT
)

ax.invert_yaxis()
ax.set_xlim(0, 205_000)
ax.set_title(
    'จำนวนโครงการตามขั้นการกำหนดกลุ่มศึกษา',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.tight_layout()

In [ ]:
png_path = figure_dir / 'fig03_04_study_scope_selection.png'
svg_path = figure_dir / 'fig03_04_study_scope_selection.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

### ผลการกำหนดขอบเขต

กลุ่มศึกษาลดจากโครงการก่อสร้างทั้งหมด 178,978 โครงการ เหลือ 136,623 โครงการที่มีวงเงินไม่เกิน 500,000 บาท และเหลือ 136,070 โครงการเมื่อจำกัดเฉพาะวิธีเฉพาะเจาะจง

การกำหนดขอบเขตนี้ไม่ได้หมายความว่า 136,070 โครงการผิดปกติ แต่ทำให้การค้นหา Pattern เปรียบเทียบโครงการที่อยู่ภายใต้เงื่อนไขกฎหมายเดียวกัน


In [ ]:
project_data.to_csv(
    project_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(f'Project rows saved: {len(project_data):,}')
print(f'Study population: {len(study_project_data):,}')
print(f'Saved to: {project_output_path}')


## 7. สรุปและคำถามส่งต่อ

สิ่งที่กฎหมายอธิบายได้คือ การใช้วิธีเฉพาะเจาะจงและการกระจุกใต้เพดาน 500,000 บาท ส่วนสิ่งที่กฎหมายยังไม่อธิบายคือ:

- เหตุใดบางคู่หน่วยงาน–ผู้รับจ้างจึงมีหลายโครงการใกล้เพดานในวันเดียวกัน
- เหตุใดบางหน่วยงานย่อยจึงพึ่งพาผู้รับจ้างรายเดียวในสัดส่วนสูง

Notebook 04 จะเริ่มจากกลุ่มศึกษา 136,070 โครงการ และตรวจสอง Pattern นี้ โดยใช้ `เลขประจำตัวนิติบุคคล 13 หลัก` เป็นคีย์ผู้รับจ้าง ส่วนชื่อผู้ชนะใช้เพื่อแสดงผลเท่านั้น
